# SCM-VLM on Kaggle (free GPU)

Before you start, set these in the right-hand panel under **Session options**:

* **Accelerator → GPU T4 x2** (or GPU P100)
* **Internet → On**. This needs a phone-verified Kaggle account.

No Kaggle dataset is needed: cell 3 downloads the data straight from the official VQA and COCO
sites.

A session lasts at most 12 hours, and only `/kaggle/working` is kept, when you click
**Save Version**. Download the results zip (last cell) before the session ends.

## 1. GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print("CUDA available:", torch.cuda.is_available())

## 2. Get the code

In [ ]:
%cd /kaggle/working
!rm -rf scm
!git clone https://github.com/MANDEEP-SAI-SURYA/SCM-VLM-Discrete-Graph.git scm
%cd /kaggle/working/scm
!git log --oneline | head -3
!pip install -q transformers timm networkx seaborn

## 3. Download the data (about 10 minutes, 1.6 GB)

This downloads the VQA v2 questions and answers, the COCO instance annotations, and a **fixed
subset of 8000 COCO images** (seed 0, so every run uses the same images). The subset has
about 43,000 questions. If anything fails, just run the cell again: files that are already
downloaded are skipped.

In [ ]:
%cd /kaggle/working/scm
!python download_data.py --images 8000
!python validate_dataset.py

## 4. Tests (CPU, about 10 s): should print `103 passed`

In [ ]:
!python -m pytest -q tests

## 5. Training runs (round 2)

Changes since round 1:

* **Discrete graph collapse fixed.** In round 1 the discrete graph was empty at test time
  (root_fraction 100 %, parents_per_node 0), so its "100 % DAG" meant nothing. A guard now keeps
  the share of "no parent" objects in the test-time graph below 50 % (`--root-target 0.5`).
* **Knowledge + symbolic fusion on the soft graph** (`soft_neurosym`). The soft graph has working
  edges and the best replacement score.
* **Seeds and significance tests.** Repeat the main runs with `--seed 1` / `--seed 2`. Section 6
  compares each run with `soft` of the same seed using the exact McNemar test.

Run the cells in order and stop when time runs out. Every finished run is usable.
Each run writes `results/eval_summary_<tag>.json`.

### Seed 42 (main runs)

In [ ]:
!python train.py --samples 30000 --epochs 10 --graph-mode soft --tag soft

In [ ]:
!python train.py --samples 30000 --epochs 10 --graph-mode soft --knowledge-prior builtin --symbolic-fusion --tag soft_neurosym

In [ ]:
!python train.py --samples 30000 --epochs 10 --graph-mode discrete --tag discrete

After the `discrete` run, check the log: `no-parent weight` should stay around or below 0.5, and
section 6 should show `parents_per_node` > 0. If you still see a collapse warning, send me the log.

### Extra seeds (to report mean ± std and test significance)

In [ ]:
!python train.py --samples 30000 --epochs 10 --graph-mode soft --seed 1 --tag soft_s1
!python train.py --samples 30000 --epochs 10 --graph-mode soft --knowledge-prior builtin --symbolic-fusion --seed 1 --tag soft_neurosym_s1

In [ ]:
!python train.py --samples 30000 --epochs 10 --graph-mode discrete --seed 1 --tag discrete_s1
!python train.py --samples 30000 --epochs 10 --graph-mode soft --seed 2 --tag soft_s2
!python train.py --samples 30000 --epochs 10 --graph-mode soft --knowledge-prior builtin --symbolic-fusion --seed 2 --tag soft_neurosym_s2

## 6. Compare the runs

The table shows mean ± std over seeds. Below it, each run is tested against `soft` of the same seed:

* difference in accuracy points, with a 95 % bootstrap CI;
* exact McNemar p-value on the same held-out questions and counterfactual examples.

Only call a difference real if p < 0.05 and the CI excludes 0.

For `soft_neurosym`, also look at **CF (neural only)** and **CF without rule answer**. The
rule engine answers existence and counting what-ifs by construction, so these two columns show
what the network itself contributes.

In [ ]:
import glob, os
tags = sorted(os.path.basename(p)[len('eval_summary_'):-5] for p in glob.glob('results/eval_summary_*.json'))
print(tags)
!python compare_runs.py --baseline soft {' '.join(tags)}

## 7. Save the results (download the zip from the file browser on the right, under /kaggle/working)

In [ ]:
%cd /kaggle/working
!zip -q -r scm_results.zip scm/results scm/checkpoints
!ls -lh scm_results.zip